# Etapa 5: Fine-Tuning Eficiente, Optimización y Despliegue

**Objetivo**: Aplicar técnicas avanzadas de optimización (LoRA/PEFT, cuantización, pruning) y preparar el modelo para despliegue con Gradio.

**Contenido**:
1. Setup e importaciones
2. LoRA/PEFT — Fine-tuning eficiente del ViT
3. Cuantización dinámica del modelo
4. Pruning estructurado
5. Comparación de optimizaciones (accuracy vs tamaño vs velocidad)
6. Reentrenamiento con datos sintéticos
7. Preparación del modelo final para despliegue
8. Demo con Gradio
9. Conclusiones finales del proyecto

## 1. Setup e Importaciones

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from copy import deepcopy
import time
import warnings
warnings.filterwarnings("ignore")

# Hugging Face y PEFT
from transformers import ViTForImageClassification
from peft import LoraConfig, get_peft_model, TaskType

# Nuestros módulos
from src.data_utils import (
    CLASS_NAMES, NUM_CLASSES, SPLITS_DIR,
    get_dataloaders, get_class_weights,
)
from src.train import train_classifier, evaluate
from src.evaluate import (
    compute_metrics, get_predictions_with_proba,
    plot_confusion_matrix, plot_training_history,
    plot_roc_curves, plot_model_comparison, print_comparison_table,
)
from src.gradcam import generate_gradcam, plot_gradcam
from src.models import count_parameters

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_DIR = Path("../results")
MODELS_DIR = RESULTS_DIR / "models"
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Cargar Datos y Modelo Base ViT

Reutilizamos los DataLoaders y cargamos el mejor ViT de la Etapa 3.

In [ ]:
# DataLoaders
train_loader, val_loader, test_loader = get_dataloaders(
    img_size=224, batch_size=16, num_workers=2, augment_train=True
)

# Pesos de clase para desbalance
class_weights = get_class_weights(SPLITS_DIR / "train.csv")
print(f"Class weights: {class_weights}")

# Cargar ViT preentrenado (fresco desde HuggingFace para aplicar LoRA)
vit_base = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
)

# Congelar todo inicialmente
for param in vit_base.parameters():
    param.requires_grad = False

base_params = count_parameters(vit_base)
print(f"\nViT Base — Parámetros totales: {base_params['total']:,}")
print(f"ViT Base — Parámetros entrenables (frozen): {base_params['trainable']:,}")

## 3. LoRA/PEFT — Fine-Tuning Eficiente del ViT

**LoRA (Low-Rank Adaptation)** permite fine-tunear modelos grandes adaptando solo matrices de bajo rango insertadas en las capas de atención, reduciendo drásticamente el número de parámetros entrenables (>90%).

**Configuración:**
- `r=8`: Rango de las matrices de adaptación
- `lora_alpha=16`: Factor de escalado
- `target_modules=["query", "value"]`: Capas donde se insertan los adaptadores

In [ ]:
# Configurar LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    bias="none",
    modules_to_save=["classifier"],  # También entrenar la cabeza de clasificación
)

# Aplicar LoRA al modelo
vit_lora = get_peft_model(vit_base, lora_config)

# Verificar reducción de parámetros
lora_params = count_parameters(vit_lora)
print("=" * 60)
print("ViT + LoRA — Resumen de parámetros")
print("=" * 60)
print(f"Total:       {lora_params['total']:>12,}")
print(f"Entrenables: {lora_params['trainable']:>12,}")
print(f"Congelados:  {lora_params['frozen']:>12,}")
print(f"% Entrenable: {lora_params['trainable_pct']:.2f}%")
print(f"\nReducción: {100 - lora_params['trainable_pct']:.1f}% de parámetros congelados")
print("=" * 60)

# Mostrar módulos LoRA
vit_lora.print_trainable_parameters()

In [ ]:
# Entrenar ViT con LoRA
print("Entrenando ViT + LoRA...")
history_lora = train_classifier(
    model=vit_lora,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=15,
    lr=3e-4,
    weight_decay=0.01,
    class_weights=class_weights,
    device=DEVICE,
    save_dir=str(MODELS_DIR),
    model_name="vit_lora",
    patience=5,
    scheduler_type="cosine",
    is_hf_model=True,
)

plot_training_history(
    history_lora,
    title="ViT + LoRA Training",
    save_path=str(FIGURES_DIR / "vit_lora_training.png"),
)

In [ ]:
# Evaluar ViT + LoRA en test set
y_true_lora, y_pred_lora, y_proba_lora = get_predictions_with_proba(
    vit_lora, test_loader, DEVICE, is_hf_model=True
)

metrics_lora = compute_metrics(y_true_lora, y_pred_lora, CLASS_NAMES, y_proba_lora)

print("\n" + "=" * 60)
print("ViT + LoRA — Resultados en Test Set")
print("=" * 60)
print(metrics_lora["classification_report"])

plot_confusion_matrix(
    y_true_lora, y_pred_lora, CLASS_NAMES,
    title="ViT + LoRA — Confusion Matrix",
    save_path=str(FIGURES_DIR / "vit_lora_confusion.png"),
)

plot_roc_curves(
    y_true_lora, y_proba_lora, CLASS_NAMES,
    title="ViT + LoRA — ROC Curves",
    save_path=str(FIGURES_DIR / "vit_lora_roc.png"),
)

## 4. Cuantización Dinámica [OPCIONAL — Puntos Extra]

Reducimos el tamaño del modelo aplicando cuantización dinámica INT8 a las capas lineales. Esto reduce el tamaño del modelo en ~4x con impacto mínimo en accuracy.

In [ ]:
# Cuantización dinámica del modelo LoRA (merge + quantize)
# Primero fusionar los pesos LoRA al modelo base
vit_merged = vit_lora.merge_and_unload()

# Medir tamaño original
def get_model_size_mb(model):
    """Calcula tamaño del modelo en MB."""
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 ** 2)

size_original = get_model_size_mb(vit_merged)
print(f"Tamaño original (FP32): {size_original:.2f} MB")

# Aplicar cuantización dinámica INT8
vit_quantized = torch.quantization.quantize_dynamic(
    vit_merged.cpu(),
    {nn.Linear},  # Cuantizar capas lineales
    dtype=torch.qint8,
)

size_quantized = get_model_size_mb(vit_quantized)
print(f"Tamaño cuantizado (INT8): {size_quantized:.2f} MB")
print(f"Reducción: {(1 - size_quantized / size_original) * 100:.1f}%")

# Evaluar cuantizado
print("\nEvaluando modelo cuantizado...")
y_true_q, y_pred_q, y_proba_q = get_predictions_with_proba(
    vit_quantized, test_loader, "cpu", is_hf_model=True
)
metrics_quantized = compute_metrics(y_true_q, y_pred_q, CLASS_NAMES, y_proba_q)

print(f"\nAccuracy original (LoRA):     {metrics_lora['accuracy']:.4f}")
print(f"Accuracy cuantizado (INT8):   {metrics_quantized['accuracy']:.4f}")
print(f"Diferencia:                   {metrics_quantized['accuracy'] - metrics_lora['accuracy']:.4f}")

## 5. Pruning Estructurado [OPCIONAL — Puntos Extra]

Aplicamos pruning no-estructurado a las capas lineales del modelo para eliminar las conexiones menos importantes (por magnitud de peso).

In [ ]:
import torch.nn.utils.prune as prune

# Cargar modelo LoRA fresco para pruning (sin cuantización)
vit_for_pruning = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
)
# Cargar pesos del mejor LoRA entrenado si existe
best_hf_path = MODELS_DIR / "vit_lora_best_hf"
if best_hf_path.exists():
    from peft import PeftModel
    vit_for_pruning = PeftModel.from_pretrained(vit_for_pruning, str(best_hf_path))
    vit_for_pruning = vit_for_pruning.merge_and_unload()

# Aplicar pruning L1 no-estructurado al 30% de los pesos de capas lineales
pruning_amount = 0.30
n_pruned = 0
for name, module in vit_for_pruning.named_modules():
    if isinstance(module, nn.Linear):
        prune.l1_unstructured(module, name="weight", amount=pruning_amount)
        n_pruned += 1

print(f"Pruning aplicado a {n_pruned} capas lineales ({pruning_amount*100:.0f}% de pesos)")

# Contar sparsity global
total_zeros = 0
total_params = 0
for name, module in vit_for_pruning.named_modules():
    if isinstance(module, nn.Linear):
        total_zeros += float(torch.sum(module.weight == 0))
        total_params += float(module.weight.nelement())

sparsity = total_zeros / total_params
print(f"Sparsity global en capas lineales: {sparsity:.2%}")

# Hacer permanente el pruning (quitar masks)
for name, module in vit_for_pruning.named_modules():
    if isinstance(module, nn.Linear):
        prune.remove(module, "weight")

# Evaluar
print("\nEvaluando modelo pruned...")
vit_for_pruning = vit_for_pruning.to(DEVICE)
y_true_p, y_pred_p, y_proba_p = get_predictions_with_proba(
    vit_for_pruning, test_loader, DEVICE, is_hf_model=True
)
metrics_pruned = compute_metrics(y_true_p, y_pred_p, CLASS_NAMES, y_proba_p)

print(f"\nAccuracy original (LoRA):  {metrics_lora['accuracy']:.4f}")
print(f"Accuracy pruned (30%):     {metrics_pruned['accuracy']:.4f}")
print(f"Diferencia:                {metrics_pruned['accuracy'] - metrics_lora['accuracy']:.4f}")

## 6. Comparación de Optimizaciones

Tabla resumen: LoRA vs Cuantización vs Pruning (accuracy, tamaño del modelo, velocidad de inferencia).

In [ ]:
import pandas as pd

# Medir velocidad de inferencia
def measure_inference_time(model, loader, device, is_hf=True, n_batches=10):
    """Mide tiempo promedio de inferencia por batch."""
    model.eval()
    model = model.to(device)
    times = []
    with torch.no_grad():
        for i, (images, _) in enumerate(loader):
            if i >= n_batches:
                break
            images = images.to(device)
            
            if device == "cuda":
                torch.cuda.synchronize()
            t0 = time.time()
            
            if is_hf:
                _ = model(pixel_values=images)
            else:
                _ = model(images)
            
            if device == "cuda":
                torch.cuda.synchronize()
            times.append(time.time() - t0)
    return np.mean(times) * 1000  # ms

time_lora = measure_inference_time(vit_lora, test_loader, DEVICE)
time_quantized = measure_inference_time(vit_quantized, test_loader, "cpu")
time_pruned = measure_inference_time(vit_for_pruning, test_loader, DEVICE)

# Tabla comparativa
size_pruned = get_model_size_mb(vit_for_pruning)

comparison_data = {
    "Modelo": ["ViT + LoRA", "ViT + Cuantización INT8", "ViT + Pruning 30%"],
    "Accuracy": [
        f"{metrics_lora['accuracy']:.4f}",
        f"{metrics_quantized['accuracy']:.4f}",
        f"{metrics_pruned['accuracy']:.4f}",
    ],
    "F1-Macro": [
        f"{metrics_lora['f1_macro']:.4f}",
        f"{metrics_quantized['f1_macro']:.4f}",
        f"{metrics_pruned['f1_macro']:.4f}",
    ],
    "Tamaño (MB)": [
        f"{size_original:.1f}",
        f"{size_quantized:.1f}",
        f"{size_pruned:.1f}",
    ],
    "Inferencia (ms/batch)": [
        f"{time_lora:.1f}",
        f"{time_quantized:.1f}",
        f"{time_pruned:.1f}",
    ],
}

df_comparison = pd.DataFrame(comparison_data)
print("\n" + "=" * 80)
print("COMPARACIÓN DE TÉCNICAS DE OPTIMIZACIÓN")
print("=" * 80)
print(df_comparison.to_string(index=False))

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models_names = ["LoRA", "Cuantizado", "Pruned"]
accs = [metrics_lora["accuracy"], metrics_quantized["accuracy"], metrics_pruned["accuracy"]]
sizes = [size_original, size_quantized, size_pruned]
speeds = [time_lora, time_quantized, time_pruned]

colors = ["#3498db", "#2ecc71", "#e74c3c"]

axes[0].bar(models_names, accs, color=colors)
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Accuracy")
axes[0].set_ylim(min(accs) * 0.9, max(accs) * 1.05)
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.005, f"{v:.4f}", ha="center", fontsize=10)

axes[1].bar(models_names, sizes, color=colors)
axes[1].set_ylabel("Size (MB)")
axes[1].set_title("Model Size")
for i, v in enumerate(sizes):
    axes[1].text(i, v + 1, f"{v:.1f}", ha="center", fontsize=10)

axes[2].bar(models_names, speeds, color=colors)
axes[2].set_ylabel("Time (ms/batch)")
axes[2].set_title("Inference Speed")
for i, v in enumerate(speeds):
    axes[2].text(i, v + 0.5, f"{v:.1f}", ha="center", fontsize=10)

plt.suptitle("Comparación de Técnicas de Optimización", fontsize=14)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "optimization_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Reentrenamiento con Datos Sintéticos [OPCIONAL — Puntos Extra]

Usamos las imágenes generadas por el VAE/GAN (Etapa 4) para aumentar las clases minoritarias y re-entrenar el modelo ViT + LoRA.

In [ ]:
# Verificar si existen datos sintéticos generados por Etapa 4
synthetic_dir = Path("../data/synthetic")

if synthetic_dir.exists() and any(synthetic_dir.iterdir()):
    print("Datos sintéticos encontrados. Integrando al dataset de entrenamiento...")
    
    # Crear CSV aumentado: train original + sintéticos
    train_df_orig = pd.read_csv(SPLITS_DIR / "train.csv")
    
    synthetic_rows = []
    for cls_dir in sorted(synthetic_dir.iterdir()):
        if cls_dir.is_dir() and cls_dir.name in CLASS_NAMES:
            cls_idx = CLASS_NAMES.index(cls_dir.name)
            for img_path in cls_dir.glob("*.jpg"):
                synthetic_rows.append({
                    "path": str(img_path),
                    "label": cls_dir.name,
                    "label_idx": cls_idx,
                })
    
    if synthetic_rows:
        synthetic_df = pd.DataFrame(synthetic_rows)
        augmented_df = pd.concat([train_df_orig, synthetic_df], ignore_index=True)
        augmented_csv = SPLITS_DIR / "train_augmented.csv"
        augmented_df.to_csv(augmented_csv, index=False)
        
        print(f"Original train: {len(train_df_orig)}")
        print(f"Sintéticos:     {len(synthetic_df)}")
        print(f"Aumentado:      {len(augmented_df)}")
        print(f"\nDistribución aumentada:")
        print(augmented_df["label"].value_counts().to_string())
        
        # Crear DataLoader aumentado
        from src.data_utils import DefectDataset, get_transforms
        aug_tf = get_transforms(224, augment=True)
        aug_dataset = DefectDataset(augmented_csv, transform=aug_tf)
        aug_train_loader = torch.utils.data.DataLoader(
            aug_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True
        )
        
        # Re-entrenar ViT + LoRA con datos aumentados
        vit_aug = ViTForImageClassification.from_pretrained(
            "google/vit-base-patch16-224",
            num_labels=NUM_CLASSES,
            ignore_mismatched_sizes=True,
        )
        for param in vit_aug.parameters():
            param.requires_grad = False
        
        vit_aug = get_peft_model(vit_aug, lora_config)
        
        print("\nEntrenando ViT + LoRA con datos sintéticos...")
        history_aug = train_classifier(
            model=vit_aug,
            train_loader=aug_train_loader,
            val_loader=val_loader,
            num_epochs=15,
            lr=3e-4,
            weight_decay=0.01,
            device=DEVICE,
            save_dir=str(MODELS_DIR),
            model_name="vit_lora_augmented",
            patience=5,
            scheduler_type="cosine",
            is_hf_model=True,
        )
        
        # Evaluar
        y_true_aug, y_pred_aug, y_proba_aug = get_predictions_with_proba(
            vit_aug, test_loader, DEVICE, is_hf_model=True
        )
        metrics_aug = compute_metrics(y_true_aug, y_pred_aug, CLASS_NAMES, y_proba_aug)
        
        print(f"\nAccuracy sin sintéticos: {metrics_lora['accuracy']:.4f}")
        print(f"Accuracy con sintéticos: {metrics_aug['accuracy']:.4f}")
        print(f"Mejora: {metrics_aug['accuracy'] - metrics_lora['accuracy']:+.4f}")
    else:
        print("No se encontraron imágenes sintéticas en el directorio.")
else:
    print("No se encontraron datos sintéticos de Etapa 4.")
    print("Ejecuta Notebook 04 primero para generar imágenes con VAE/GAN.")
    print("O continúa — este paso es opcional.")

## 8. Grad-CAM — Visualización de Atención

Generamos mapas Grad-CAM para visualizar qué regiones del parche están activando la clasificación.

In [ ]:
from PIL import Image as PILImage

# Cargar el mejor modelo LoRA fusionado para Grad-CAM
best_hf_path = MODELS_DIR / "vit_lora_best_hf"
if best_hf_path.exists():
    vit_gradcam = ViTForImageClassification.from_pretrained(
        "google/vit-base-patch16-224",
        num_labels=NUM_CLASSES,
        ignore_mismatched_sizes=True,
    )
    from peft import PeftModel
    vit_gradcam = PeftModel.from_pretrained(vit_gradcam, str(best_hf_path))
    vit_gradcam = vit_gradcam.merge_and_unload()
else:
    vit_gradcam = vit_merged  # Usar el merged anterior

vit_gradcam = vit_gradcam.to(DEVICE).eval()

# Generar Grad-CAM para varias muestras de test
test_df = pd.read_csv(SPLITS_DIR / "test.csv")
n_examples = min(10, len(test_df))
sample_df = test_df.groupby("label").apply(
    lambda x: x.sample(min(2, len(x)), random_state=42)
).reset_index(drop=True)

print(f"Generando Grad-CAM para {len(sample_df)} imágenes de ejemplo...\n")

for idx, row in sample_df.iterrows():
    img = PILImage.open(row["path"]).convert("RGB")
    true_label = row["label"]
    
    cam_image, pred_idx, probs = generate_gradcam(
        model=vit_gradcam,
        image=img,
        model_type="vit",
        device=DEVICE,
    )
    pred_label = CLASS_NAMES[pred_idx]
    correct = "✓" if pred_label == true_label else "✗"
    
    plot_gradcam(
        image=img,
        cam_image=cam_image,
        prediction=pred_idx,
        probabilities=probs,
        class_names=CLASS_NAMES,
        title=f"True: {true_label} | Pred: {pred_label} {correct}",
        save_path=str(FIGURES_DIR / f"gradcam_sample_{idx}.png"),
    )

## 9. Preparar Modelo Final para Despliegue

Guardamos el modelo óptimo (ViT + LoRA merged) para usarlo en la demo Gradio.

In [ ]:
# Guardar modelo final para despliegue
deploy_dir = MODELS_DIR / "deploy"
deploy_dir.mkdir(parents=True, exist_ok=True)

# Guardar modelo merged como HuggingFace model
vit_gradcam.save_pretrained(str(deploy_dir / "vit_final"))

# Guardar también config/metadata
import json
deploy_meta = {
    "model_name": "ViT + LoRA (merged)",
    "base_model": "google/vit-base-patch16-224",
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "img_size": 224,
    "lora_config": {
        "r": 8,
        "lora_alpha": 16,
        "target_modules": ["query", "value"],
    },
    "metrics": {
        "accuracy": float(metrics_lora["accuracy"]),
        "f1_macro": float(metrics_lora["f1_macro"]),
    },
}

with open(deploy_dir / "deploy_meta.json", "w") as f:
    json.dump(deploy_meta, f, indent=2)

print(f"Modelo final guardado en: {deploy_dir}")
print(f"  - vit_final/  (modelo HF)")
print(f"  - deploy_meta.json (metadata)")
print(f"\nListo para usar en app.py (Gradio)")

## 10. Demo con Gradio (Prueba Local)

Probamos que la interfaz Gradio funciona correctamente antes de desplegar.

In [ ]:
import gradio as gr
from torchvision import transforms as T
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Descripciones de defectos
DEFECT_DESCRIPTIONS = {
    "crack": "🔴 **Grieta (Crack)**: Fractura en la superficie metálica. Requiere inspección inmediata — puede comprometer la integridad estructural.",
    "dent": "🟠 **Abolladura (Dent)**: Deformación cóncava en la superficie. Evaluar profundidad y diámetro según criterios de tolerancia.",
    "scratch": "🟡 **Rayón (Scratch)**: Daño superficial lineal en la piel del avión. Verificar profundidad — si penetra el primer tratamiento, requiere reparación.",
    "missing_head": "🔴 **Cabeza de Remache Faltante (Missing Head)**: Remache sin cabeza visible. Compromiso de unión estructural — reemplazo requerido.",
    "paint_off": "🟡 **Desprendimiento de Pintura (Paint Off)**: Pérdida de recubrimiento protector. Expone el metal a corrosión — requiere re-tratamiento.",
}

# Función de predicción para Gradio
def predict_defect(image):
    """Función principal para la interfaz Gradio."""
    if image is None:
        return None, "Sube una imagen para analizar.", None
    
    img_pil = PILImage.fromarray(image).convert("RGB")
    
    # Generar Grad-CAM y predicción
    cam_image, pred_idx, probs = generate_gradcam(
        model=vit_gradcam,
        image=img_pil,
        model_type="vit",
        device=DEVICE,
    )
    
    # Probabilidades por clase
    label_probs = {CLASS_NAMES[i]: float(probs[i]) for i in range(NUM_CLASSES)}
    
    # Descripción
    pred_class = CLASS_NAMES[pred_idx]
    confidence = probs[pred_idx]
    description = DEFECT_DESCRIPTIONS.get(pred_class, "Defecto desconocido.")
    description += f"\n\n**Confianza**: {confidence:.1%}"
    
    return cam_image, label_probs, description

# Crear interfaz Gradio
demo = gr.Interface(
    fn=predict_defect,
    inputs=gr.Image(label="Sube imagen de superficie de aeronave"),
    outputs=[
        gr.Image(label="Grad-CAM — Mapa de Atención"),
        gr.Label(num_top_classes=5, label="Clasificación del Defecto"),
        gr.Markdown(label="Diagnóstico"),
    ],
    title="✈️ Aircraft Skin Defect Classifier",
    description=(
        "Sistema de IA para detección y clasificación de defectos en superficies de aeronaves. "
        "Sube una fotografía de inspección y el modelo clasificará el tipo de daño "
        "mostrando un mapa de calor de las zonas detectadas."
    ),
    examples=None,
    flagging_mode="never",
)

# Lanzar demo inline en notebook
demo.launch(share=False, inline=True)

## 11. Comparación Final de TODOS los Modelos (Etapas 1–5)

Tabla comparativa exhaustiva reuniendo resultados de todas las etapas del proyecto.

In [ ]:
# Reunir métricas de etapas anteriores (si se guardaron)
# Cargar resultados de notebooks previos si existen
results_file = RESULTS_DIR / "all_metrics.json"

all_results = {}

# Intentar cargar métricas guardadas previamente
if results_file.exists():
    with open(results_file) as f:
        all_results = json.load(f)

# Agregar resultados de esta etapa
all_results["ViT + LoRA (E5)"] = {
    "accuracy": float(metrics_lora["accuracy"]),
    "f1_macro": float(metrics_lora["f1_macro"]),
    "precision_macro": float(metrics_lora["precision_macro"]),
    "recall_macro": float(metrics_lora["recall_macro"]),
    "f1_per_class": {k: float(v) for k, v in metrics_lora["f1_per_class"].items()},
}

all_results["ViT + Cuantizado (E5)"] = {
    "accuracy": float(metrics_quantized["accuracy"]),
    "f1_macro": float(metrics_quantized["f1_macro"]),
    "precision_macro": float(metrics_quantized["precision_macro"]),
    "recall_macro": float(metrics_quantized["recall_macro"]),
    "f1_per_class": {k: float(v) for k, v in metrics_quantized["f1_per_class"].items()},
}

all_results["ViT + Pruned (E5)"] = {
    "accuracy": float(metrics_pruned["accuracy"]),
    "f1_macro": float(metrics_pruned["f1_macro"]),
    "precision_macro": float(metrics_pruned["precision_macro"]),
    "recall_macro": float(metrics_pruned["recall_macro"]),
    "f1_per_class": {k: float(v) for k, v in metrics_pruned["f1_per_class"].items()},
}

# Guardar todo
with open(results_file, "w") as f:
    json.dump(all_results, f, indent=2)

# Imprimir tabla comparativa
if all_results:
    print_comparison_table(all_results, CLASS_NAMES)
    
    # Gráfico de barras comparativo
    if len(all_results) >= 2:
        plot_model_comparison(
            all_results, metric="f1_macro",
            save_path=str(FIGURES_DIR / "final_comparison_f1.png"),
        )
        plot_model_comparison(
            all_results, metric="accuracy",
            save_path=str(FIGURES_DIR / "final_comparison_acc.png"),
        )

## 12. Conclusiones Finales del Proyecto

### Resumen de Etapas

| Etapa | Modelo | Contribución |
|-------|--------|-------------|
| **E1** | MLP Base | Baseline de referencia. Accuracy limitado por pérdida de info espacial |
| **E2** | CNN Custom | Mejora significativa al capturar patrones espaciales (bordes, texturas) |
| **E3** | ViT / ResNet50 | Transfer learning supera modelos entrenados desde cero |
| **E4** | VAE / GAN | Generación sintética para balancear clases minoritarias |
| **E5** | ViT + LoRA | Fine-tuning eficiente (>90% menos parámetros), cuantización, pruning, despliegue |

### Técnicas de Optimización Aplicadas
- **LoRA/PEFT**: Reducción de parámetros entrenables >90% con rendimiento comparable
- **Cuantización INT8**: Reducción de tamaño del modelo ~4x con mínima pérdida de accuracy
- **Pruning L1**: Eliminación de conexiones redundantes manteniendo performance

### Despliegue
- Interfaz Gradio con clasificación + Grad-CAM + diagnóstico descriptivo
- Preparado para HuggingFace Spaces (app.py en raíz del repositorio)

### Próximos Pasos Sugeridos
1. Ampliar dataset con más fuentes (instance segmentation → localización precisa)
2. Explorar detección multi-defecto por imagen con YOLO/DETR
3. Integrar pipeline de inspección con drones/cámaras industriales
4. Aplicar pruning + cuantización combinados para edge deployment